In [29]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [30]:
class BatsmanState(TypedDict):

    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [37]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])*100

    state['sr'] = sr

    return {'sr': sr}

In [32]:
def calculate_bpb(state: BatsmanState):
    
    bpb = state['balls']/(state['fours'] + state['sixes'])

    state['bpb'] = bpb

    return {'bpb': bpb}

In [38]:
def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    boundary_percent = (((state['fours'] * 4) + (state['sixes'] * 6))/state['runs'])*100

    state['boundary_percent'] = boundary_percent
    return {'boundary_percent': boundary_percent}

In [42]:
def summary(state: BatsmanState) -> BatsmanState:
    
    summary = f"""
strike rate - {state['sr']} \n
Balls per boundary - {state['bpb']} \n
Boundary percent - {state['boundary_percent']}
 
"""
    
    state['summary'] = summary

    return {'summary': summary}

In [43]:
# create graph

graph = StateGraph(BatsmanState)

graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_sr')
graph.add_edge(START, 'calculate_bpb')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_sr', 'summary')
graph.add_edge('calculate_bpb', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()


In [44]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(initial_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nstrike rate - 200.0 \n\nBalls per boundary - 5.0 \n\nBoundary percent - 48.0\n \n'}